# Stage 6b — Alternative denominators and time-to-event (post hoc; DEVIATIONS.md D7)

Post hoc response to the critique that the persistent-predicate proportion is not a standardized
probability of post-recall reuse. Uses exactly the Stage 6 definitions:

* **Edges:** Rule 2 v2 (`data/predicate_edges.csv`).
* **Post-recall citation:** a non-recalled device whose 510(k) was **received after the predicate's
  recall was public** (`data/recall_public_dates.csv`, written by Stage 6).
* **Evaluable recalled devices:** those with a valid recall date (Stage 5) and a public date.

Time-to-event starts at the public date and ends at the first post-recall citing submission's receipt
date. Follow-up runs to the last clearance in the corpus (administrative censoring; competing risks are
not modelled because public data hold no event that ends predicate eligibility).

Writes `data/denominators.csv`, `data/km_cumulative_incidence.csv`, `data/matched_comparator.csv` and
`data/denominators_summary.json`. Network: none. Run after Stage 6.

In [1]:
import gzip, json
import numpy as np
import pandas as pd
import networkx as nx

corpus = pd.read_csv("data/corpus.csv", dtype=str)
corpus["decision_year"] = pd.to_numeric(corpus["decision_year"], errors="coerce")
corpus["dd"] = pd.to_datetime(corpus["decision_date"], errors="coerce")
edges = pd.read_csv("data/predicate_edges.csv", dtype=str)
recalled = pd.read_csv("data/recalled_nodes.csv", dtype=str)
public = pd.read_csv("data/recall_public_dates.csv", dtype=str)
manifest = pd.read_csv("data/download_manifest.csv", dtype=str)
EXPECTED = json.load(open("data/expected_values.json"))
assert EXPECTED.get("post_recall_rule") == "received_after_public", "run Stage 6 (2026-10 version) first"

received = {r["k_number"]: pd.to_datetime(r.get("date_received"), errors="coerce")
            for r in json.load(gzip.open("snapshot/510k_raw.json.gz", "rt"))}
code = dict(zip(corpus["k_number"], corpus["product_codes"]))
cyear = dict(zip(corpus["k_number"], corpus["decision_year"]))
recall_date = dict(zip(recalled["k_number"], pd.to_datetime(recalled["event_date_initiated"], errors="coerce")))
pub = dict(zip(public["k_number"], pd.to_datetime(public["recall_public_date"])))
recalled_set = set(recalled["k_number"]) & set(corpus["k_number"])
has_summary = set(manifest.loc[manifest["status"] == "200", "k_number"])
END = corpus["dd"].max()                      # last clearance observable in the corpus

G = nx.DiGraph(); G.add_nodes_from(corpus["k_number"])
G.add_edges_from((a, b) for a, b in zip(edges["predicate_knumber"], edges["device_knumber"]) if a in G and b in G)
print(f"CHECKPOINT  end of observation {END.date()} | recalled devices {len(recalled_set)}")

CHECKPOINT  end of observation 2026-07-24 | recalled devices 1691


### Per-device outcome, follow-up and prior use

In [2]:
rx = {k: received.get(k, pd.NaT) for k in corpus["k_number"]}
same_code = corpus.assign(rx=corpus["k_number"].map(rx)).groupby("product_codes")
rows = []
for A in sorted(recalled_set):
    P = pub.get(A, pd.NaT)
    d = dict(k=A, code=code[A], cyear=cyear[A], P=P)
    if pd.isna(recall_date.get(A)) or pd.isna(P):
        d["evaluable"] = False; rows.append(d); continue
    succ = list(G.successors(A))
    post = sorted(rx[B] for B in succ if B not in recalled_set and pd.notna(rx[B]) and rx[B] > P)
    prior = any(pd.notna(rx[B]) and rx[B] < P for B in succ)
    fam = same_code.get_group(code[A])
    later = fam[(fam["rx"] > P) & (fam["k_number"] != A)]
    d.update(evaluable=True, persist=bool(post), prior=prior, ever_cited=bool(succ),
             t_event=(post[0] - P).days / 365.25 if post else np.nan,
             fu=max((END - P).days / 365.25, 0.0),
             later_same_sum=int(later["k_number"].isin(has_summary).sum()),
             last_same=fam.loc[fam["k_number"] != A, "dd"].max())
    rows.append(d)
df = pd.DataFrame(rows)
E = df[df["evaluable"]].copy()
for c in ["persist", "prior", "ever_cited"]:
    E[c] = E[c].astype(bool)
n_persist = int(E["persist"].sum())
print(f"CHECKPOINT  evaluable {len(E)} of {len(df)} | persistent {n_persist} (Stage 6: {EXPECTED['persistent_predicates']})")
assert n_persist == EXPECTED["persistent_predicates"], "6b does not reproduce Stage 6's persistent-predicate count"

CHECKPOINT  evaluable 1689 of 1691 | persistent 893 (Stage 6: 893)


### Table A — persistent recalled predicates under alternative denominators (Wilson 95% CI)

In [3]:
def wilson(k, n, z=1.959964):
    p = k / n; den = 1 + z * z / n; c = (p + z * z / (2 * n)) / den
    h = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5) / den
    return c - h, c + h

def row(label, s):
    n = len(s); k = int(s["persist"].sum())
    lo, hi = wilson(k, n) if n else (np.nan, np.nan)
    return dict(population=label, n=n, persistent=k, pct=round(100 * k / n, 1) if n else np.nan,
                ci_lo=round(100 * lo, 1), ci_hi=round(100 * hi, 1),
                median_fu_y=round(s["fu"].median(), 1) if n else np.nan)

T = pd.DataFrame([
    row("All evaluable recalled devices", E),
    row(">=3 y follow-up after public recall", E[E["fu"] >= 3]),
    row(">=5 y follow-up after public recall", E[E["fu"] >= 5]),
    row(">=10 y follow-up after public recall", E[E["fu"] >= 10]),
    row("Prior predicate use (cited by a 510(k) received before public recall)", E[E["prior"]]),
    row("No prior predicate use", E[~E["prior"]]),
    row("Prior use, >=5 y follow-up", E[E["prior"] & (E["fu"] >= 5)]),
    row("No prior use, >=5 y follow-up", E[~E["prior"] & (E["fu"] >= 5)]),
    row(">=1 later same-code 510(k) with a retrievable summary", E[E["later_same_sum"] > 0]),
    row("No later same-code 510(k) with a retrievable summary", E[E["later_same_sum"] == 0]),
    row("Recall public in 2026 (little observable follow-up)", E[E["P"].dt.year == 2026]),
])
T.to_csv("data/denominators.csv", index=False)
print(T.to_string(index=False))

                                                           population    n  persistent  pct  ci_lo  ci_hi  median_fu_y
                                       All evaluable recalled devices 1689         893 52.9   50.5   55.2         10.9
                                  >=3 y follow-up after public recall 1489         855 57.4   54.9   59.9         12.0
                                  >=5 y follow-up after public recall 1374         805 58.6   56.0   61.2         12.4
                                 >=10 y follow-up after public recall  953         595 62.4   59.3   65.5         14.0
Prior predicate use (cited by a 510(k) received before public recall) 1002         565 56.4   53.3   59.4         10.1
                                               No prior predicate use  687         328 47.7   44.0   51.5         11.9
                                           Prior use, >=5 y follow-up  790         503 63.7   60.3   67.0         12.1
                                        No prior

### Table B — cumulative incidence of first post-recall citation (1 − Kaplan–Meier, log-log 95% CI)

In [4]:
def km(t, e):
    t = np.asarray(t, float); e = np.asarray(e, bool); o = np.argsort(t); t, e = t[o], e[o]
    S, V, out = 1.0, 0.0, []
    for u in np.unique(t[e]):
        n = (t >= u).sum(); d = ((t == u) & e).sum()
        S *= 1 - d / n; V += d / (n * (n - d)) if n > d else 0; out.append((u, S, V))
    return out

def at(out, x):
    S, V = 1.0, 0.0
    for u, s, v in out:
        if u <= x: S, V = s, v
    if S in (0, 1): return round(100 * (1 - S), 1), None, None
    se = np.sqrt(V) / abs(np.log(S)); lo = S ** np.exp(1.96 * se); hi = S ** np.exp(-1.96 * se)
    return round(100 * (1 - S), 1), round(100 * (1 - lo), 1), round(100 * (1 - hi), 1)

def run(label, s, opportunity=False):
    e = s["persist"].to_numpy(bool)
    t = np.where(e, s["t_event"], s["fu"]).astype(float)
    if opportunity:   # also censor at the last clearance in the device's own product code
        c = ((pd.to_datetime(s["last_same"]) - s["P"]).dt.days / 365.25).clip(lower=0).fillna(0).to_numpy()
        late = e & (s["t_event"].to_numpy() > c); e = e & ~late
        t = np.where(e, s["t_event"], np.minimum(s["fu"], c)).astype(float)
    o = km(t, e); r = {"cohort": label, "n": len(s), "events": int(e.sum())}
    for y in (1, 2, 3, 5, 10, 15):
        p, lo, hi = at(o, y); r[f"{y}y"] = f"{p} ({hi}-{lo})" if lo is not None else f"{p}"
    return r

K = [run("All", E), run("Prior predicate use", E[E["prior"]]), run("No prior predicate use", E[~E["prior"]]),
     run("All - also censored at last same-code clearance", E, opportunity=True)]
for era, (a, b) in {"Public <=2005": (1900, 2005), "Public 2006-2015": (2006, 2015), "Public 2016-2026": (2016, 2026)}.items():
    K.append(run(era, E[E["P"].dt.year.between(a, b)]))
K = pd.DataFrame(K); K.to_csv("data/km_cumulative_incidence.csv", index=False)
print(K.to_string(index=False))
med_t = float(E.loc[E["persist"], "t_event"].median())
print(f"CHECKPOINT  median time to first post-recall citing submission (among events) {med_t:.1f} y")

                                         cohort    n  events               1y               2y               3y               5y              10y              15y
                                            All 1689     893 20.2 (18.4-22.2) 30.3 (28.1-32.6) 37.4 (35.1-39.8) 46.1 (43.6-48.6) 57.6 (55.0-60.2) 60.9 (58.1-63.7)
                            Prior predicate use 1002     565 23.8 (21.3-26.6) 35.7 (32.7-38.8) 43.1 (40.0-46.3) 52.3 (49.1-55.6) 62.3 (58.9-65.7) 65.3 (61.7-69.0)
                         No prior predicate use  687     328 14.9 (12.4-17.9) 22.5 (19.5-25.9) 29.2 (25.8-32.8) 37.1 (33.5-41.0) 50.6 (46.7-54.8) 54.5 (50.3-58.8)
All - also censored at last same-code clearance 1689     870 20.4 (18.5-22.4) 30.4 (28.2-32.8) 37.6 (35.2-40.0) 46.3 (43.9-48.9) 58.3 (55.6-61.0) 61.9 (59.0-64.8)
                                  Public <=2005   76      48  14.5 (8.3-24.6) 25.0 (16.7-36.3) 25.0 (16.7-36.3) 34.2 (24.7-46.0) 53.9 (43.3-65.4) 56.6 (45.8-67.8)
                      

### Table C — exploratory benchmark: matched non-recalled devices
Each evaluable recalled device is matched to all non-recalled devices with the same product code and
clearance year. Comparators take the recalled device's **public date** as their index date and the same
outcome is applied (a non-recalled 510(k) citing them, received after the index date). A second set also
matches on prior-use status. Bootstrap (2,000 resamples of recalled devices, seed 1) for the difference.
Not causal: recalls concentrate in commercially active device lines.

In [5]:
succ = {}
for a, b in zip(edges["predicate_knumber"], edges["device_knumber"]):
    succ.setdefault(a, []).append(b)
nonrec = corpus[~corpus["k_number"].isin(recalled_set)]
pools = nonrec.groupby(["product_codes", "decision_year"])["k_number"].apply(list).to_dict()
def outcome(C, idx): return any(B not in recalled_set and pd.notna(rx.get(B)) and rx[B] > idx for B in succ.get(C, []))
def prior_use(C, idx): return any(pd.notna(rx.get(B)) and rx[B] < idx for B in succ.get(C, []))
res = []
for _, a in E.iterrows():
    pool = pools.get((a["code"], a["cyear"]), [])
    if not pool: continue
    o = np.mean([outcome(c, a["P"]) for c in pool])
    pm = [c for c in pool if prior_use(c, a["P"]) == a["prior"]]
    res.append(dict(k=a["k"], recalled_persist=a["persist"], prior=a["prior"], comparator_pct=o,
                    comparator_prior_matched_pct=np.mean([outcome(c, a["P"]) for c in pm]) if pm else np.nan,
                    n_pool=len(pool)))
X = pd.DataFrame(res); X.to_csv("data/matched_comparator.csv", index=False)
Y = X.dropna(subset=["comparator_prior_matched_pct"])
rng = np.random.default_rng(1); boot = []
for _ in range(2000):
    s = Y.sample(len(Y), replace=True, random_state=int(rng.integers(1_000_000_000)))
    boot.append(s["recalled_persist"].mean() - s["comparator_prior_matched_pct"].mean())
diff = 100 * (Y["recalled_persist"].mean() - Y["comparator_prior_matched_pct"].mean())
lo, hi = np.percentile(np.array(boot) * 100, [2.5, 97.5])
print(f"CHECKPOINT  matched on code+year: n {len(X)} | recalled {100*X['recalled_persist'].mean():.1f}% vs comparators {100*X['comparator_pct'].mean():.1f}%")
print(f"CHECKPOINT  also matched on prior use: n {len(Y)} | recalled {100*Y['recalled_persist'].mean():.1f}% vs "
      f"comparators {100*Y['comparator_prior_matched_pct'].mean():.1f}% | difference {diff:+.1f} pts (95% CI {lo:.1f} to {hi:.1f})")

CHECKPOINT  matched on code+year: n 1591 | recalled 53.6% vs comparators 38.2%
CHECKPOINT  also matched on prior use: n 1519 | recalled 53.3% vs comparators 40.3% | difference +13.0 pts (95% CI 10.5 to 15.6)


In [6]:
summary = {
    "post_recall_rule": EXPECTED["post_recall_rule"], "end_of_observation": str(END.date()),
    "evaluable": int(len(E)), "persistent": n_persist,
    "pct_all": float(T.loc[0, "pct"]), "pct_fu5": float(T.loc[2, "pct"]),
    "pct_prior": float(T.loc[4, "pct"]), "pct_no_prior": float(T.loc[5, "pct"]),
    "km_all": K.loc[0, ["1y", "3y", "5y", "10y"]].to_dict(), "km_events": int(K.loc[0, "events"]),
    "median_time_to_event_y": round(med_t, 2),
    "comparator_prior_matched": {"n": int(len(Y)), "recalled_pct": round(100 * Y["recalled_persist"].mean(), 1),
                                 "comparator_pct": round(100 * Y["comparator_prior_matched_pct"].mean(), 1),
                                 "difference_pts": round(diff, 1), "ci": [round(lo, 1), round(hi, 1)]},
}
json.dump(summary, open("data/denominators_summary.json", "w"), indent=2)
print("CHECKPOINT  wrote data/denominators.csv, km_cumulative_incidence.csv, matched_comparator.csv, denominators_summary.json")

CHECKPOINT  wrote data/denominators.csv, km_cumulative_incidence.csv, matched_comparator.csv, denominators_summary.json
